# Read Data from silver container

In [0]:
df = spark.read.format("delta").load("abfss://silver@adlscarsales.dfs.core.windows.net/carSales/")
display(df)

In [0]:
from pyspark.sql.functions import trim, col
df = df.withColumn("Date_ID", trim(col("Date_ID")))

## Create a temporary view to perform sql operations

In [0]:
df.createOrReplaceTempView("sales")

In [0]:
df_src = spark.sql("""
          select Date_ID, max(Date) as Date
          from sales
          group by Date_ID
          """)
display(df_src)

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import monotonically_increasing_id, cast

In [0]:
if spark.catalog.tableExists("carsalescatalog.gold.dim_date"):
    df_sink = spark.sql("""
                        select dim_date_key, Date_ID, Date
                        from carsalescatalog.gold.dim_date
                        """)
    
else:
    df_sink = spark.sql("""
          select 1 as dim_date_key, Date_ID, Date
          from sales
          where 1=0
          """)
    
    

In [0]:
df_sink.display()

In [0]:
df_new_data = df_src.join(df_sink, df_src["Date_ID"] == df_sink["Date_ID"], "left")\
    .select(df_src["Date_ID"], df_src["Date"], df_sink["dim_date_key"])

In [0]:
df_new_records = df_new_data.filter(df_new_data.dim_date_key.isNull())
df_old_records = df_new_data.filter(df_new_data.dim_date_key.isNotNull())

# Add surrogate dim key for new records

In [0]:
if spark.catalog.tableExists("carsalescatalog.gold.dim_date"):
    max_value = spark.sql("""
                        select max(dim_date_key)
                        from carsalescatalog.gold.dim_date
                        """).collect()[0][0]
    
else:
    max_value = 0

In [0]:
df_new_records = df_new_records.withColumn("dim_date_key", max_value + monotonically_increasing_id() + 1)
display(df_new_records)

# Appending new and old records after adding surrogate key

In [0]:
df_final = df_new_records.unionByName(df_old_records)
display(df_final)

# Update Date dimension table

In [0]:
if DeltaTable.isDeltaTable(spark, "abfss://gold@adlscarsales.dfs.core.windows.net/dim_date/"):
    deltatable = DeltaTable.forPath(spark, "abfss://gold@adlscarsales.dfs.core.windows.net/dim_date/")

    deltatable.alias("trg").merge(df_final.alias("src"), "trg.Date_ID = src.Date_ID")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()

else:
    df_final.write.format("delta").mode("overwrite").save("abfss://gold@adlscarsales.dfs.core.windows.net/dim_date/")

    spark.sql("""
              create table carsalescatalog.gold.dim_date
              using delta
              location 'abfss://gold@adlscarsales.dfs.core.windows.net/dim_date/'
              """)

In [0]:
spark.read.format("delta").load("abfss://gold@adlscarsales.dfs.core.windows.net/dim_date/").display()